PySpark Deep Dive - Interview Preparation Guide

# Part 1: Core Concepts 核心概念


## 1. Spark Execution Flow | Spark 执行流程

### 🎤 English Answer (30-40s)
"When you submit a Spark application, the Driver program creates a SparkContext and connects to the Cluster Manager. The Driver converts your code into a logical plan, then a physical plan with stages and tasks. The Cluster Manager allocates Executors across worker nodes. Tasks are distributed to Executors, which process data partitions in parallel. Results flow back to the Driver. The key flow is: Driver → Cluster Manager → Executors → Tasks on Partitions."

### 📖 中文详解
**执行流程分解：**

1. **Driver Program 启动**
   - 创建 SparkContext/SparkSession
   - 解析用户代码，构建 logical plan（逻辑计划）

2. **DAG Scheduler 工作**
   - 将 logical plan 转换为 DAG（有向无环图）
   - 根据 shuffle 边界划分 Stages

3. **Task Scheduler 分配**
   - 将每个 Stage 拆分为 Tasks
   - 每个 partition 对应一个 Task

4. **Cluster Manager 协调**
   - 可以是 YARN、Kubernetes、Standalone
   - 负责资源分配和 Executor 启动

5. **Executor 执行**
   - 在 Worker 节点上运行
   - 执行具体的 Task，处理数据分区

```
用户代码 → Driver（解析）→ DAG → Stages → Tasks → Executors（执行）→ 结果返回
```

---



## 2. DAG (Directed Acyclic Graph) | 有向无环图

### 🎤 English Answer (30-40s)
"DAG stands for Directed Acyclic Graph. In Spark, the DAG represents the sequence of computations on your data. Each node is an RDD or DataFrame operation, and edges show data dependencies. 'Directed' means data flows one way, 'Acyclic' means no loops. The DAG Scheduler uses this graph to optimize execution by combining operations and determining stage boundaries at shuffle points. This enables Spark's lazy evaluation and whole-stage code generation."

### 📖 中文详解
**DAG 的核心概念：**

- **Directed（有向）**：数据流动有明确方向，从源到结果
- **Acyclic（无环）**：不存在循环依赖，避免无限循环
- **Graph（图）**：节点代表 RDD/DataFrame，边代表转换操作

**为什么需要 DAG？**

1. **优化执行计划**：可以合并多个操作（pipeline）
2. **容错恢复**：知道数据血缘（lineage），可重算丢失分区
3. **延迟执行**：只有 action 触发时才真正执行

```python
# 示例：这段代码会形成一个 DAG
df.filter(col("age") > 25)      # 节点1：Filter
  .groupBy("city")               # 节点2：GroupBy（触发 shuffle）
  .count()                       # 节点3：Aggregation
  .show()                        # Action：触发执行
```

---



## 3. Wide vs Narrow Dependencies | 宽依赖 vs 窄依赖

### 🎤 English Answer (30-40s)
"Narrow dependency means each parent partition is used by at most one child partition - like map, filter, or union. Data stays local, no shuffle needed. Wide dependency means a parent partition is used by multiple child partitions - like groupBy, join, or reduceByKey. This requires shuffle, moving data across the network. Stage boundaries are drawn at wide dependencies. Narrow transformations can be pipelined together for efficiency, while wide dependencies are expensive and often cause performance bottlenecks."

### 📖 中文详解

| 特性 | 窄依赖 (Narrow) | 宽依赖 (Wide) |
|------|-----------------|---------------|
| **数据流动** | 一对一或多对一 | 一对多 |
| **Shuffle** | 不需要 | 需要 |
| **网络传输** | 无 | 大量 |
| **Stage 划分** | 同一 Stage 内 | Stage 边界 |
| **容错代价** | 低（只重算单个分区）| 高（可能重算多个分区）|

**窄依赖操作示例：**
```python
# 这些操作都是窄依赖，可以 pipeline
df.filter(col("status") == "active")  # filter
  .select("name", "age")               # select/map
  .withColumn("age_plus", col("age") + 1)  # map
```

**宽依赖操作示例：**
```python
# 这些操作触发 shuffle，是 Stage 边界
df.groupBy("department").agg(sum("salary"))  # groupBy
df1.join(df2, "id")                           # join
df.repartition(100)                           # repartition
df.orderBy("date")                            # sort
```

---



## 4. Shuffle | 数据重分布

### 🎤 English Answer (30-40s)
"Shuffle is Spark's mechanism for redistributing data across partitions, triggered by wide transformations like groupBy or join. During shuffle, map tasks write intermediate data to local disk, organized by target partition. Then reduce tasks fetch this data over the network. Shuffle is expensive because it involves disk I/O, serialization, and network transfer. It can cause spill to disk if memory is insufficient. Optimizing shuffle - through broadcast joins, proper partitioning, or salting for skew - is critical for Spark performance."

### 📖 中文详解

**Shuffle 的两个阶段：**

1. **Shuffle Write（Map 端）**
   - 将数据按目标分区 key 进行分组
   - 写入本地磁盘的 shuffle 文件
   - 生成 index 文件记录每个分区的位置

2. **Shuffle Read（Reduce 端）**
   - 从所有 Map 任务拉取属于自己分区的数据
   - 可能需要排序和聚合
   - 如果内存不足会 spill 到磁盘

**Shuffle 为什么慢？**
```
1. 磁盘 I/O：写入和读取 shuffle 文件
2. 网络传输：跨节点数据移动
3. 序列化/反序列化：数据格式转换
4. 可能的 spill：内存不足时写磁盘
```

**Shuffle 相关配置：**
```python
spark.conf.set("spark.sql.shuffle.partitions", 200)  # shuffle 后的分区数
spark.conf.set("spark.shuffle.compress", "true")      # 压缩 shuffle 数据
spark.conf.set("spark.shuffle.spill.compress", "true") # 压缩 spill 数据
```

---



## 5. Catalyst Optimizer | 查询优化器

### 🎤 English Answer (30-40s)
"Catalyst is Spark SQL's query optimizer. It takes your DataFrame or SQL query and applies rule-based and cost-based optimizations. The process has four phases: Analysis resolves column names and types, Logical Optimization applies rules like predicate pushdown and column pruning, Physical Planning generates multiple execution strategies and picks the best one, and Code Generation creates optimized JVM bytecode. Catalyst is why DataFrames often outperform hand-written RDD code - it automatically optimizes your queries."

### 📖 中文详解

**Catalyst 四个优化阶段：**

```
未解析的逻辑计划 → 分析 → 逻辑优化 → 物理计划 → 代码生成
   (Unresolved)    (Analysis) (Logical)  (Physical)  (CodeGen)
```

1. **Analysis（分析）**
   - 解析表名、列名
   - 验证数据类型
   - 使用 Catalog 获取 schema 信息

2. **Logical Optimization（逻辑优化）**
   - **Predicate Pushdown**：把 filter 下推到数据源
   - **Column Pruning**：只读取需要的列
   - **Constant Folding**：预计算常量表达式
   - **Combine Filters**：合并多个 filter

3. **Physical Planning（物理计划）**
   - 生成多个可能的执行计划
   - 使用 cost model 选择最优方案
   - 决定 join 策略（broadcast vs sort-merge）

4. **Code Generation（代码生成）**
   - 使用 Tungsten 的 whole-stage codegen
   - 生成优化的 JVM 字节码

```python
# 查看执行计划
df.explain(True)  # 显示所有阶段的计划
```

---



## 6. Tungsten | 内存与执行优化

### 🎤 English Answer (30-40s)
"Tungsten is Spark's execution engine focused on CPU and memory efficiency. It has three main features: First, off-heap memory management that bypasses JVM garbage collection and uses direct binary representation. Second, cache-aware computation that optimizes for CPU cache locality. Third, whole-stage code generation that compiles query plans into single optimized functions, eliminating virtual function calls. Tungsten is why Spark 2.0+ is much faster than earlier versions - it operates closer to bare metal performance."

### 📖 中文详解

**Tungsten 三大优化：**

1. **内存管理（Memory Management）**
   - 使用 off-heap 内存，避免 GC 开销
   - 直接操作二进制数据，减少序列化
   - 紧凑的数据布局，减少内存占用

2. **缓存感知计算（Cache-aware Computation）**
   - 优化数据结构以利用 CPU L1/L2/L3 缓存
   - 减少 cache miss，提高 CPU 利用率

3. **Whole-Stage Code Generation（全阶段代码生成）**
   - 将多个操作编译成单个 Java 函数
   - 消除虚函数调用开销
   - 利用 CPU 流水线和 SIMD

```python
# Tungsten 相关配置
spark.conf.set("spark.sql.codegen.wholeStage", "true")  # 启用全阶段代码生成
```

**传统执行 vs Tungsten：**
```
传统：filter() → 虚函数调用 → map() → 虚函数调用 → 聚合
Tungsten：编译成单个优化函数，直接执行
```

---



## 7. Broadcast Join | 广播连接

### 🎤 English Answer (30-40s)
"Broadcast join is used when joining a large table with a small table. Instead of shuffling both tables, Spark broadcasts the small table to all executors, keeping it in memory. Each executor then joins its local partitions of the large table with the broadcasted data - no shuffle needed for the large table. Use broadcast join when one table fits in memory, typically under 10MB by default but configurable. It dramatically reduces shuffle overhead and is one of the most effective join optimizations."

### 📖 中文详解

**Broadcast Join 原理：**
```
普通 Join：大表 shuffle + 小表 shuffle = 大量网络传输
Broadcast Join：小表广播到所有节点 + 大表本地 join = 最小网络传输
```

**使用场景：**
- 一个表很小（默认 < 10MB，可配置）
- 另一个表很大
- 想避免 shuffle

**代码示例：**
```python
from pyspark.sql.functions import broadcast

# 方法1：显式使用 broadcast hint
result = large_df.join(broadcast(small_df), "key")

# 方法2：配置自动广播阈值
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 100 * 1024 * 1024)  # 100MB

# 方法3：SQL hint
spark.sql("""
    SELECT /*+ BROADCAST(small_table) */ *
    FROM large_table JOIN small_table ON large_table.id = small_table.id
""")
```

**注意事项：**
- 广播表必须能放进 Driver 内存
- 广播表也必须能放进每个 Executor 内存
- 过大的广播会导致 OOM

---



## 8. Data Skew | 数据倾斜

### 🎤 English Answer (30-40s)
"Data skew happens when some partitions have much more data than others, causing few tasks to run much longer. Solutions include: First, salting - add random prefix to skewed keys, join with exploded dimension table, then aggregate. Second, broadcast join if one table is small enough. Third, AQE's skew join optimization that automatically splits skewed partitions. Fourth, isolate skewed keys - process them separately and union results. Fifth, increase parallelism to distribute data more evenly. Diagnosing skew through Spark UI task duration variance is the first step."

### 📖 中文详解

**什么是数据倾斜？**
- 某些 key 的数据量远超其他 key
- 导致个别 Task 处理时间远超平均
- 整个 Job 等待最慢的 Task

**诊断方法：**
1. Spark UI 查看 Task 执行时间分布
2. 检查 shuffle read/write 数据量差异
3. 统计 key 分布：`df.groupBy("key").count().orderBy(desc("count"))`

**解决方案：**

### 方案1：Salting（加盐）
```python
from pyspark.sql.functions import concat, lit, rand, floor, explode, array

# 对大表的 skewed key 加随机后缀
salt_count = 10
large_df_salted = large_df.withColumn(
    "salted_key", 
    concat(col("key"), lit("_"), floor(rand() * salt_count))
)

# 对小表扩展 salt_count 倍
small_df_exploded = small_df.withColumn(
    "salt", 
    explode(array([lit(i) for i in range(salt_count)]))
).withColumn(
    "salted_key",
    concat(col("key"), lit("_"), col("salt"))
)

# 用 salted_key join
result = large_df_salted.join(small_df_exploded, "salted_key")
```

### 方案2：隔离倾斜 Key
```python
# 分离处理
skewed_keys = ["hot_key_1", "hot_key_2"]

# 倾斜数据单独处理（broadcast join）
skewed_df = large_df.filter(col("key").isin(skewed_keys))
skewed_result = skewed_df.join(broadcast(small_df), "key")

# 正常数据正常处理
normal_df = large_df.filter(~col("key").isin(skewed_keys))
normal_result = normal_df.join(small_df, "key")

# 合并结果
final_result = skewed_result.union(normal_result)
```

### 方案3：AQE 自动处理
```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", 5)
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256MB")
```

---



## 9. RDD vs DataFrame | 弹性分布式数据集 vs 数据框

### 🎤 English Answer (30-40s)
"RDD is Spark's fundamental abstraction - a distributed collection of objects with explicit transformations. You control everything but get no optimization. DataFrame is a distributed table with named columns and a schema. The key difference is optimization: DataFrames go through Catalyst optimizer and Tungsten execution engine, making them typically 10x faster. DataFrames also provide a unified API across languages. Use RDDs only when you need low-level control or work with unstructured data. For most analytics, DataFrames are preferred."

### 📖 中文详解

| 特性 | RDD | DataFrame |
|------|-----|-----------|
| **抽象层次** | 低级（对象集合）| 高级（表格结构）|
| **Schema** | 无，自己定义类型 | 有，命名列和类型 |
| **优化** | 无自动优化 | Catalyst + Tungsten |
| **性能** | 较慢 | 通常快 10x |
| **序列化** | Java 序列化 | Tungsten 二进制格式 |
| **语言一致性** | 不同语言 API 不同 | 统一 API |
| **使用场景** | 非结构化数据、细粒度控制 | 结构化/半结构化数据分析 |

```python
# RDD 操作示例
rdd = sc.parallelize([("a", 1), ("b", 2)])
result = rdd.map(lambda x: (x[0], x[1] * 2)).reduceByKey(lambda a, b: a + b)

# DataFrame 操作示例（更简洁，自动优化）
df = spark.createDataFrame([("a", 1), ("b", 2)], ["key", "value"])
result = df.groupBy("key").agg(sum("value") * 2)
```

**何时用 RDD？**
- 需要细粒度控制每个元素的处理
- 处理非结构化数据（如图数据）
- 需要使用 RDD 专有操作（如 aggregate、fold）

---



## 10. SparkSession | Spark 会话

### 🎤 English Answer (30-40s)
"SparkSession is the unified entry point for Spark since version 2.0. It combines SparkContext, SQLContext, and HiveContext into one object. You create it with SparkSession.builder(), configure settings, and use it to read data, create DataFrames, run SQL queries, and access the underlying SparkContext. It manages the connection to the cluster and holds all runtime configurations. In a typical application, you create one SparkSession at the start and use it throughout. It's the first thing you initialize in any Spark program."

### 📖 中文详解

**SparkSession 的演进：**
```
Spark 1.x：SparkContext + SQLContext + HiveContext（分散）
Spark 2.0+：SparkSession（统一入口）
```

**创建和配置：**
```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyApp") \
    .master("yarn") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.sql.shuffle.partitions", "200") \
    .enableHiveSupport() \
    .getOrCreate()

# 获取 SparkContext（如果需要 RDD 操作）
sc = spark.sparkContext

# 读取数据
df = spark.read.parquet("path/to/data")

# 执行 SQL
spark.sql("SELECT * FROM table")

# 创建 DataFrame
df = spark.createDataFrame(data, schema)
```

**重要属性：**
- `spark.sparkContext`：底层 SparkContext
- `spark.catalog`：管理表和数据库
- `spark.conf`：运行时配置
- `spark.udf`：注册 UDF

---



## 11. Spark SQL

### 🎤 English Answer (30-40s)
"Spark SQL is Spark's module for structured data processing. It allows you to run SQL queries directly on DataFrames and integrate with Hive. You can mix SQL with DataFrame operations - register a DataFrame as a temp view, query it with SQL, and continue processing with DataFrame API. Spark SQL uses Catalyst optimizer for query optimization and supports standard SQL syntax. It's great for analysts familiar with SQL and enables seamless integration with BI tools through JDBC/ODBC. The same execution engine handles both SQL and DataFrame operations."

### 📖 中文详解

**使用方式：**
```python
# 方法1：直接在 DataFrame 上使用 SQL 函数
from pyspark.sql.functions import col, sum, avg

df.filter(col("age") > 25) \
  .groupBy("department") \
  .agg(avg("salary").alias("avg_salary"))

# 方法2：注册临时视图，写 SQL
df.createOrReplaceTempView("employees")
result = spark.sql("""
    SELECT department, AVG(salary) as avg_salary
    FROM employees
    WHERE age > 25
    GROUP BY department
""")

# 方法3：访问 Hive 表
spark.sql("SELECT * FROM hive_database.table_name")
```

**Spark SQL 优势：**
1. **熟悉的语法**：标准 SQL，学习曲线低
2. **优化执行**：共享 Catalyst 优化器
3. **互操作性**：SQL 和 DataFrame 可混用
4. **Hive 兼容**：支持 HiveQL 和 Hive 元数据

---



## 12. Stage vs Task vs Job | 阶段 vs 任务 vs 作业

### 🎤 English Answer (30-40s)
"In Spark's execution hierarchy: A Job is triggered by an action like collect() or save(). Each Job is divided into Stages at shuffle boundaries - operations before a shuffle form one stage. Within each Stage, there are Tasks - one Task per partition. So if you have 100 partitions and 3 stages, you'll have 300 tasks total. Tasks within a stage can run in parallel. Stages must run sequentially because later stages depend on shuffle output from earlier stages. Understanding this helps you optimize parallelism and diagnose bottlenecks in Spark UI."

### 📖 中文详解

**层级关系：**
```
Application（应用）
    └── Job（作业）：由 action 触发
           └── Stage（阶段）：由 shuffle 划分
                  └── Task（任务）：每个 partition 一个 task
```

**具体示例：**
```python
# 这段代码会产生什么？
df.filter(col("status") == "active")    # Stage 1: 窄依赖操作
  .groupBy("category")                   # Stage 1 结束，Stage 2 开始（shuffle）
  .agg(count("*"))                       # Stage 2
  .orderBy(desc("count"))                # Stage 2 结束，Stage 3 开始（shuffle）
  .show()                                # Action，触发 Job

# 结果：1 个 Job，3 个 Stages
# 如果原始数据有 100 个 partitions：
# Stage 1: 100 个 tasks
# Stage 2: 200 个 tasks（默认 shuffle partitions）
# Stage 3: 200 个 tasks
```

**Spark UI 中的观察：**
- **Jobs 页面**：显示所有 Jobs 及状态
- **Stages 页面**：显示每个 Stage 的 tasks 数量和时间
- **Tasks 页面**：显示每个 Task 的详细指标

---



## 13. Persist vs Cache | 持久化 vs 缓存

### 🎤 English Answer (30-40s)
"Cache and persist both store DataFrames in memory for reuse, avoiding recomputation. cache() is shorthand for persist(MEMORY_AND_DISK). persist() lets you choose storage level - MEMORY_ONLY, MEMORY_AND_DISK, DISK_ONLY, or with serialization and replication options. Use caching when you'll reuse a DataFrame multiple times, especially after expensive operations. But don't over-cache - it consumes memory needed for execution. Always unpersist() when done. Check Spark UI Storage tab to monitor cached data. Remember, caching is lazy - it happens on first action."

### 📖 中文详解

**Storage Levels（存储级别）：**

| 级别 | 描述 | 使用场景 |
|------|------|----------|
| MEMORY_ONLY | 仅内存，不够则重算 | 数据小，内存充足 |
| MEMORY_AND_DISK | 内存优先，溢出到磁盘 | 默认推荐（cache()）|
| DISK_ONLY | 仅磁盘 | 数据大，重算代价高 |
| MEMORY_ONLY_SER | 内存，序列化存储 | 节省内存，CPU 换空间 |
| MEMORY_AND_DISK_SER | 内存+磁盘，序列化 | 平衡方案 |
| *_2 | 带副本（如 MEMORY_AND_DISK_2） | 容错要求高 |

**代码示例：**
```python
from pyspark import StorageLevel

# 简单缓存
df.cache()  # 等同于 df.persist(StorageLevel.MEMORY_AND_DISK)

# 指定存储级别
df.persist(StorageLevel.MEMORY_ONLY_SER)

# 触发缓存（缓存是 lazy 的）
df.count()  # 这时才真正缓存

# 检查是否已缓存
df.is_cached

# 释放缓存
df.unpersist()
```

**最佳实践：**
1. **何时缓存**：DataFrame 被多次使用（如迭代算法、多路输出）
2. **何时不缓存**：只用一次、数据太大、内存紧张
3. **监控缓存**：Spark UI → Storage 标签页

---



## 14. Accumulator | 累加器

### 🎤 English Answer (30-40s)
"Accumulators are shared variables for aggregating information across executors back to the driver. Common uses include counting events, summing values, or tracking errors during processing. They're write-only on executors - workers can only add to them, not read. The driver reads the final value after an action completes. Spark guarantees accuracy only in actions, not transformations - in transformations, updates might happen multiple times due to task retries. Built-in accumulators support Long, Double, and collections. You can also create custom accumulators."

### 📖 中文详解

**Accumulator 特点：**
- Executor 只能写（add），不能读
- Driver 可以读取最终值
- 用于收集分布式计算中的聚合信息

**使用场景：**
1. 统计处理的记录数
2. 统计错误或异常数量
3. 跟踪特定条件的出现次数

**代码示例：**
```python
# 创建累加器
error_count = spark.sparkContext.accumulator(0)
processed_count = spark.sparkContext.accumulator(0)

def process_row(row):
    processed_count.add(1)
    try:
        # 处理逻辑
        return process(row)
    except Exception:
        error_count.add(1)
        return None

# 在 RDD 操作中使用
rdd.foreach(process_row)

# 在 Driver 中读取结果
print(f"Processed: {processed_count.value}")
print(f"Errors: {error_count.value}")
```

**注意事项：**
```python
# ⚠️ 在 transformation 中，由于 task 重试，可能重复计数
rdd.map(lambda x: (error_count.add(1), x)).collect()  # 不可靠

# ✅ 在 action 中使用更可靠
rdd.foreach(lambda x: error_count.add(1))  # 可靠
```

---



## 15. UDF (User Defined Function) | 用户自定义函数

### 🎤 English Answer (30-40s)
"UDFs let you extend Spark SQL with custom Python functions. You define a Python function, register it with a return type, then use it in DataFrame operations or SQL queries. However, UDFs have performance overhead - data must be serialized between JVM and Python, and they prevent Catalyst optimization. Prefer built-in functions when possible. For better performance, use Pandas UDFs which process batches using Apache Arrow for faster serialization. Pandas UDFs can be scalar, grouped map, or grouped aggregate types depending on your use case."

### 📖 中文详解

**普通 UDF：**
```python
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, IntegerType

# 定义函数
def categorize_age(age):
    if age < 18:
        return "minor"
    elif age < 65:
        return "adult"
    else:
        return "senior"

# 注册为 UDF
categorize_udf = udf(categorize_age, StringType())

# 使用 UDF
df.withColumn("age_category", categorize_udf(col("age")))

# 或注册到 SQL
spark.udf.register("categorize_age", categorize_age, StringType())
spark.sql("SELECT name, categorize_age(age) as category FROM people")
```

**Pandas UDF（性能更好）：**
```python
from pyspark.sql.functions import pandas_udf
import pandas as pd

# Scalar Pandas UDF：一列输入，一列输出
@pandas_udf(StringType())
def categorize_age_pandas(ages: pd.Series) -> pd.Series:
    return ages.apply(lambda x: "minor" if x < 18 else "adult" if x < 65 else "senior")

df.withColumn("category", categorize_age_pandas(col("age")))
```

**性能对比：**
```
普通 UDF：每行数据 JVM ↔ Python 序列化
Pandas UDF：批量处理，使用 Arrow 格式，快 10-100x
```

**最佳实践：**
1. 优先使用内置函数（最快）
2. 需要自定义时，优先用 Pandas UDF
3. 普通 UDF 作为最后选择

---



## 16. Transformations | 转换操作

### 🎤 English Answer (30-40s)
"Transformations in Spark are lazy operations that define a computation but don't execute immediately. They're divided into narrow and wide. Narrow transformations like map, filter, and select process partitions independently - no data movement needed. Wide transformations like groupBy, join, and repartition require shuffle - data moves across partitions. This distinction matters because narrow transformations can be pipelined together in one stage, while each wide transformation creates a new stage. Understanding this helps you minimize shuffles and optimize performance."

### 📖 中文详解

**Narrow Transformations（窄转换）示例：**
```python
# 这些操作可以在同一 Stage 中 pipeline
df.select("col1", "col2")           # 选择列
df.filter(col("age") > 25)          # 过滤行
df.withColumn("new", col("a") + 1)  # 添加/修改列
df.drop("col")                      # 删除列
df.distinct()                       # 去重（同分区内）
df.sample(0.1)                      # 采样
df.union(df2)                       # 合并（分区不变）
```

**Wide Transformations（宽转换）示例：**
```python
# 这些操作触发 shuffle，产生新 Stage
df.groupBy("key").agg(...)          # 分组聚合
df.join(df2, "key")                 # Join
df.repartition(100)                 # 重分区
df.orderBy("col")                   # 全局排序
df.distinct()                       # 全局去重
df.coalesce(10)                     # 减少分区（可能 shuffle）
```

**优化思路：**
```python
# ❌ 不好：多次 shuffle
df.groupBy("a").count().orderBy("count").filter(col("count") > 10)

# ✅ 更好：先 filter 减少数据量
df.filter(col("count") > 10).groupBy("a").count().orderBy("count")

# ❌ 不好：多次 join
result = df1.join(df2, "key").join(df3, "key").join(df4, "key")

# ✅ 更好：broadcast 小表
result = df1.join(broadcast(df2), "key").join(broadcast(df3), "key")
```

---



# Part 2: 必讲清楚的核心机制

---



## Spark Task Execution Flow | Task 执行流程

### 🎤 English Answer (30-40s)
"When a Task executes on an Executor: First, the TaskScheduler sends serialized Task to an Executor. The Executor deserializes it and creates a TaskContext. For DataFrame operations, Tungsten's code-generated functions process data in batches. Data is read from source or shuffle files into memory. Narrow transformations are pipelined together. Results are either returned to Driver, written to storage, or written as shuffle files for the next stage. Each Task processes one partition independently. Task metrics like shuffle read/write and spill are tracked and reported back to Driver."

### 📖 中文详解

**Task 执行详细步骤：**

```
1. Driver 发送 Task
   └── 序列化 Task（包含代码和元数据）
   └── 通过网络发送到 Executor

2. Executor 接收 Task
   └── 反序列化 Task
   └── 创建 TaskContext（存储运行时信息）

3. Task 执行
   └── 读取数据（从源文件或 shuffle 文件）
   └── 应用 pipelined transformations
   └── Tungsten 生成的代码处理数据批次
   
4. Task 完成
   └── 输出结果（返回 Driver / 写文件 / 写 shuffle 文件）
   └── 报告 metrics 给 Driver
   └── 释放资源
```

**Task 关键指标（Spark UI 可见）：**
- **Duration**：执行时间
- **GC Time**：垃圾回收时间
- **Shuffle Read**：读取的 shuffle 数据量
- **Shuffle Write**：写入的 shuffle 数据量
- **Spill (Memory)**：内存溢出量
- **Spill (Disk)**：磁盘溢出量

---



## Stage Division Principle | Stage 划分原理

### 🎤 English Answer (30-40s)
"Stage division follows a simple rule: work backwards from the final RDD, and draw a stage boundary at every wide dependency. All narrow dependencies stay in the same stage and get pipelined together. The DAGScheduler creates a DAG of stages where parent stages must complete before child stages can run - because child stages need shuffle output from parents. Within a stage, all tasks can run in parallel since they're independent. This design maximizes parallelism within stages while respecting data dependencies between stages."

### 📖 中文详解

**划分规则：**
```python
# 从最终 RDD 向前回溯
# 遇到宽依赖就划分新 Stage

df.filter(...)       # ─┐
  .select(...)       #  │ Stage 1（窄依赖，pipeline）
  .withColumn(...)   # ─┘
  .groupBy(...)      # ══ shuffle 边界 ══
  .agg(...)          # ─┐ Stage 2
  .orderBy(...)      # ══ shuffle 边界 ══
  .limit(10)         # ─  Stage 3
```

**Stage 依赖关系：**
```
Stage 1 (ShuffleMapStage)
    ↓ shuffle
Stage 2 (ShuffleMapStage)
    ↓ shuffle  
Stage 3 (ResultStage)
```

**为什么这样设计？**
1. **最大化并行**：Stage 内的 Tasks 可以完全并行
2. **最小化等待**：只在 shuffle 点同步
3. **优化 pipeline**：窄依赖操作合并执行，减少 I/O

---



## Memory Model | 内存模型

### 🎤 English Answer (30-40s)
"Spark's memory is divided into Execution and Storage, managed by the Unified Memory Manager. Execution memory handles shuffles, joins, sorts, and aggregations - the actual computation. Storage memory caches RDDs and DataFrames, plus broadcast variables. By default, they share the heap with a soft boundary at 50-50. When execution needs more memory and storage isn't full, it can borrow from storage. Storage can also borrow from unused execution memory but must give it back when needed. This unified model maximizes memory utilization while prioritizing execution."

### 📖 中文详解

**内存划分：**
```
Executor Memory (spark.executor.memory)
├── Reserved Memory (~300MB)
│   └── Spark 内部使用
├── User Memory (1 - spark.memory.fraction)
│   └── 用户数据结构、UDF 变量
└── Spark Memory (spark.memory.fraction, 默认 0.6)
    ├── Execution Memory (spark.memory.storageFraction 的剩余)
    │   └── Shuffle、Join、Sort、Aggregation
    └── Storage Memory (spark.memory.storageFraction, 默认 0.5)
        └── Cache、Broadcast 变量
```

**内存借用机制：**
```
┌─────────────────────────────────────────┐
│           Unified Memory Pool           │
│  ┌─────────────┬─────────────────────┐  │
│  │  Execution  │      Storage        │  │
│  │   (动态)    │←→    (动态)         │  │
│  └─────────────┴─────────────────────┘  │
└─────────────────────────────────────────┘

规则：
1. Execution 可以借用 Storage 空间（可驱逐 cache）
2. Storage 可以借用空闲 Execution 空间
3. Execution 优先级更高，需要时可收回借出的内存
```

**关键配置：**
```python
spark.conf.set("spark.executor.memory", "8g")           # Executor 总内存
spark.conf.set("spark.memory.fraction", "0.6")          # Spark 管理的比例
spark.conf.set("spark.memory.storageFraction", "0.5")   # Storage 初始比例
spark.conf.set("spark.memory.offHeap.enabled", "true")  # 启用堆外内存
spark.conf.set("spark.memory.offHeap.size", "4g")       # 堆外内存大小
```

---



# Part 3: 性能调优必会

---



## How to Tune Spark Jobs | 如何调优

### 🎤 English Answer (30-40s)
"Spark tuning follows a systematic approach: First, check Spark UI for bottlenecks - long tasks, high GC, shuffle spill. Second, right-size partitions - aim for 100-200MB each. Third, optimize shuffles - use broadcast joins for small tables, repartition strategically. Fourth, manage memory - balance execution vs storage, tune GC settings. Fifth, enable AQE for dynamic optimization. Sixth, fix data skew through salting or skew join hints. Finally, tune parallelism - match cores and partitions. Always profile before and after changes to measure impact."

### 📖 中文详解

**调优检查清单：**

### 1. 检查 Spark UI
```
- Jobs 页面：哪些 jobs 慢？
- Stages 页面：哪个 stage 是瓶颈？
- Tasks 页面：task 时间分布是否均匀？（skew 检测）
- Storage 页面：缓存使用情况
- Executors 页面：GC 时间、内存使用
```

### 2. 分区调优
```python
# 读取时指定分区
df = spark.read.parquet("path").repartition(200)

# shuffle 分区数
spark.conf.set("spark.sql.shuffle.partitions", 200)

# 目标：每个分区 100-200MB
# 计算：数据总量 / 目标分区大小 = 分区数
```

### 3. Join 优化
```python
# 小表广播
result = large_df.join(broadcast(small_df), "key")

# 调整广播阈值
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 100 * 1024 * 1024)
```

### 4. 内存调优
```python
spark.conf.set("spark.executor.memory", "8g")
spark.conf.set("spark.executor.memoryOverhead", "2g")  # 堆外内存
spark.conf.set("spark.memory.fraction", "0.6")
```

### 5. 启用 AQE
```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```

---



## Why is the Job Slow? | 为什么 Job 很慢

### 🎤 English Answer (30-40s)
"Common reasons for slow Spark jobs: Data skew - few tasks take much longer, check task duration variance. Too many or too few partitions - wrong parallelism. Excessive shuffle - check shuffle read/write sizes, consider broadcast joins. Memory pressure - high GC time, shuffle spill to disk. Inefficient operations - UDFs instead of built-in functions, unnecessary collects. Resource contention - not enough executors or cores. I/O bottleneck - slow storage or small files problem. Always start diagnosis with Spark UI to identify which stage and what metric is the bottleneck."

### 📖 中文详解

**诊断流程：**

```
Job 慢？
│
├── 检查 Spark UI
│   ├── 某个 Stage 特别慢？
│   │   ├── Tasks 时间差异大 → 数据倾斜
│   │   ├── Shuffle Read/Write 大 → Shuffle 瓶颈
│   │   └── 所有 Tasks 都慢 → 操作本身慢
│   │
│   ├── GC Time 高？ → 内存不足
│   ├── Spill 到磁盘？ → 增加内存或减少分区
│   └── Executor 失败重试？ → OOM 或不稳定
│
└── 检查代码
    ├── 使用了 UDF？ → 改用内置函数
    ├── collect() 大数据？ → 避免全量收集
    └── 多次重复计算？ → 添加 cache()
```

**常见原因和解决方案：**

| 症状 | 可能原因 | 解决方案 |
|------|----------|----------|
| 个别 Task 很慢 | 数据倾斜 | Salting / AQE |
| 所有 Task 都慢 | 分区太大 | 增加分区数 |
| 大量 Shuffle | 不必要的宽依赖 | Broadcast join |
| 高 GC 时间 | 内存不足 | 增加内存 / 减少 cache |
| Spill 到磁盘 | Execution 内存不足 | 调整 memory.fraction |
| 小文件太多 | Input 碎片化 | Coalesce 或合并文件 |

---



## How Many Partitions | 分区数量

### 🎤 English Answer (30-40s)
"The optimal partition count depends on your data size and cluster resources. Rule of thumb: aim for 100-200MB per partition. For shuffle operations, spark.sql.shuffle.partitions defaults to 200, but adjust based on data volume. Too few partitions means underutilized parallelism and possible OOM. Too many means excessive scheduling overhead and small files. Calculate as: total data size divided by target partition size. Also consider: 2-4 partitions per CPU core for good parallelism. With AQE enabled, Spark can automatically coalesce small partitions, which helps."

### 📖 中文详解

**分区计算公式：**
```python
# 理想分区数 = 数据总量 / 目标分区大小
# 例如：100GB 数据，目标每分区 200MB
num_partitions = 100 * 1024 / 200 = 512 个分区

# 另一个考虑：每个 CPU core 2-4 个分区
# 例如：10 个 Executor，每个 4 cores = 40 cores
# 分区数 = 40 * 3 = 120 个分区（取中间值）
```

**配置方式：**
```python
# 读取时指定分区
df = spark.read.option("maxPartitionsPerTrigger", 100).parquet("path")

# Shuffle 后的分区数
spark.conf.set("spark.sql.shuffle.partitions", 200)

# 文件源的默认分区
spark.conf.set("spark.sql.files.maxPartitionBytes", 128 * 1024 * 1024)

# AQE 自动合并小分区
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128MB")
```

**检查当前分区数：**
```python
df.rdd.getNumPartitions()
```

---



## Coalesce vs Repartition | 合并 vs 重分区

### 🎤 English Answer (30-40s)
"Both coalesce and repartition change partition count, but differently. Coalesce reduces partitions without full shuffle - it just combines existing partitions on the same executors, so it's more efficient for reducing partitions. Repartition does a full shuffle and can increase or decrease partitions evenly. Use coalesce when reducing partitions, especially before writing to avoid small files. Use repartition when you need to increase partitions or need even data distribution. Repartition can take a column argument for partitioning by key. Remember, coalesce can result in uneven partitions."

### 📖 中文详解

| 特性 | coalesce(n) | repartition(n) |
|------|-------------|----------------|
| **分区变化** | 只能减少 | 增加或减少 |
| **Shuffle** | 避免（通常）| 完全 shuffle |
| **数据分布** | 可能不均匀 | 均匀分布 |
| **性能** | 更快（无 shuffle）| 较慢（有 shuffle）|
| **使用场景** | 减少分区、写文件前 | 需要均匀分布时 |

**代码示例：**
```python
# coalesce：减少分区，无 shuffle
df_200_partitions.coalesce(10).write.parquet("output")  # 写前减少分区

# repartition：增加分区，有 shuffle
df_10_partitions.repartition(100)  # 需要更多并行度

# repartition by column：按列分区
df.repartition(100, "date")  # 相同 date 的数据在同一分区

# repartition + sort within partitions
df.repartition("date").sortWithinPartitions("timestamp")
```

**常见错误：**
```python
# ❌ 用 coalesce 增加分区（无效）
df.coalesce(1000)  # 如果原本只有 100 分区，还是 100

# ❌ 写文件前 repartition（浪费）
df.repartition(10).write.parquet("output")  # 不如用 coalesce

# ✅ 正确做法
df.coalesce(10).write.parquet("output")
```

---



## Shuffle Spill Control | 控制 Shuffle Spill

### 🎤 English Answer (30-40s)
"Shuffle spill happens when execution memory is insufficient to hold shuffle data, forcing writes to disk - this is expensive. To control it: First, increase executor memory or execution memory fraction. Second, reduce partition size by increasing partition count. Third, use serialized storage to reduce memory footprint. Fourth, avoid unnecessary wide transformations. Monitor spill in Spark UI's stage details - both memory spill and disk spill metrics. If you see significant spill, either give more memory or reduce the data processed per task by using more partitions."

### 📖 中文详解

**Spill 发生原因：**
```
Shuffle 过程中，数据需要在内存中排序/聚合
如果 Execution Memory 不足 → 数据溢出到磁盘 → Spill
```

**Spill 的代价：**
- 额外的磁盘 I/O
- 序列化/反序列化开销
- 显著降低性能

**解决方案：**

### 1. 增加内存
```python
spark.conf.set("spark.executor.memory", "16g")
spark.conf.set("spark.memory.fraction", "0.8")  # 更多给 Spark 管理
```

### 2. 增加分区数（减小每个 Task 处理的数据量）
```python
spark.conf.set("spark.sql.shuffle.partitions", 400)  # 从 200 增加到 400
```

### 3. 减少缓存，留更多给 Execution
```python
spark.conf.set("spark.memory.storageFraction", "0.3")  # 减少 Storage 比例
# 或者 unpersist 不需要的缓存
df.unpersist()
```

### 4. 启用压缩（减少内存占用）
```python
spark.conf.set("spark.shuffle.compress", "true")
spark.conf.set("spark.shuffle.spill.compress", "true")
```

**监控 Spill：**
```
Spark UI → Stages → 点击具体 Stage → 查看 Task 详情
关注指标：
- Spill (Memory)：溢出前在内存中的大小
- Spill (Disk)：写入磁盘的大小
```

---



## Cluster Mode vs Client Mode | 集群模式 vs 客户端模式

### 🎤 English Answer (30-40s)
"The key difference is where the Driver runs. In Client Mode, the Driver runs on the machine that submits the job - good for interactive development and debugging because you see output directly. In Cluster Mode, the Driver runs inside the cluster on one of the worker nodes - better for production because if your client machine disconnects, the job continues. Use Client Mode for development, spark-shell, and notebooks. Use Cluster Mode for production jobs submitted via spark-submit. The Executors run on worker nodes in both modes."

### 📖 中文详解

| 特性 | Client Mode | Cluster Mode |
|------|-------------|--------------|
| **Driver 位置** | 提交机器上 | 集群 Worker 节点上 |
| **输出查看** | 直接在终端看到 | 需要查看日志 |
| **网络依赖** | 客户端必须保持连接 | 提交后可断开 |
| **使用场景** | 开发、调试、交互式 | 生产环境 |
| **资源使用** | 客户端需要足够资源 | 集群自己管理 |

**提交命令示例：**
```bash
# Client Mode
spark-submit \
  --master yarn \
  --deploy-mode client \
  --executor-memory 4g \
  my_script.py

# Cluster Mode
spark-submit \
  --master yarn \
  --deploy-mode cluster \
  --executor-memory 4g \
  my_script.py
```

**何时用哪种？**
```
Client Mode：
- spark-shell / pyspark 交互式
- Jupyter Notebook
- 开发调试阶段

Cluster Mode：
- 生产环境定时任务
- 长时间运行的 Job
- 需要容错的场景
```

---



## Lazy Evaluation | 惰性求值

### 🎤 English Answer (30-40s)
"Lazy evaluation means transformations don't execute immediately - Spark just builds a logical plan. Execution only happens when you call an action like collect, count, show, or write. This design enables optimization: Catalyst can see the entire plan and optimize it, combine operations, push down filters, prune columns before any data moves. It also means if you define transformations but never call an action, nothing actually happens. Understanding this helps you write efficient code - define all transformations first, let Spark optimize, then trigger with one action."

### 📖 中文详解

**Transformation vs Action：**

| Transformation（惰性）| Action（触发执行）|
|-----------------------|-------------------|
| filter, select, join | collect, count |
| groupBy, withColumn | show, take |
| map, flatMap | write, save |
| union, distinct | foreach |
| orderBy, limit | first, head |

**示例：**
```python
# 这些操作不会执行任何计算
df_filtered = df.filter(col("age") > 25)        # 只是记录计划
df_grouped = df_filtered.groupBy("city")         # 继续记录
df_result = df_grouped.agg(count("*"))           # 还是没执行

# 直到这里才真正执行
df_result.show()  # Action! 触发整个计划的执行
```

**为什么要惰性求值？**

1. **优化机会**
```python
# Spark 可以优化这个计划
df.select("*").filter(col("age") > 25).select("name")
# 优化为：只读取 age 和 name 列，filter 下推
```

2. **避免不必要计算**
```python
# 如果后面没有 action，前面的工作都不会做
df.filter(...).groupBy(...).agg(...)  # 什么都不会发生
```

3. **Pipeline 执行**
```python
# 多个窄依赖操作可以合并成一个 Task
df.filter(...).select(...).withColumn(...)  # 一次遍历完成
```

---



## Data Broadcasting | 数据广播

### 🎤 English Answer (30-40s)
"Broadcast variables let you efficiently share read-only data across all executors. Instead of shipping data with each task, Spark sends it once per executor using a peer-to-peer protocol. Common uses include lookup tables, ML model parameters, or configuration data. For joins, broadcast join sends the small table to all executors, avoiding shuffle of the large table. Use broadcast for data that fits in executor memory and is accessed by many tasks. Call broadcast() explicitly or configure auto-broadcast threshold. Remember to unpersist when done to free memory."

### 📖 中文详解

**Broadcast 变量：**
```python
# 创建广播变量
lookup_dict = {"A": 1, "B": 2, "C": 3}
broadcast_lookup = spark.sparkContext.broadcast(lookup_dict)

# 在 UDF 或 RDD 操作中使用
def lookup_value(key):
    return broadcast_lookup.value.get(key, 0)

# 释放
broadcast_lookup.unpersist()
```

**Broadcast Join：**
```python
from pyspark.sql.functions import broadcast

# 显式广播小表
result = large_df.join(broadcast(small_df), "key")

# 配置自动广播阈值
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 100 * 1024 * 1024)  # 100MB
```

**Broadcast 原理：**
```
普通变量：每个 Task 都会收到一份（N 个 Task = N 份数据）
Broadcast 变量：每个 Executor 只收到一份（M 个 Executor = M 份数据）

数据分发方式：BitTorrent-like 协议
Driver → Executor1 → Executor2 → Executor3 ...
            ↓           ↓
         Executor4   Executor5
```

---



## Data Salting | 数据加盐

### 🎤 English Answer (30-40s)
"Salting is a technique to handle data skew by adding random values to skewed keys. For a join with skew: First, identify skewed keys. Second, salt the large table by appending random numbers 0 to N to the key. Third, explode the small table N times, each with a different salt value. Fourth, join on the salted key. This distributes the skewed key's data across N partitions instead of one. After joining, you can remove the salt column. Salting trades some extra computation for much better parallelism, and is very effective for severe skew that AQE can't handle."

### 📖 中文详解

**完整 Salting 实现：**
```python
from pyspark.sql.functions import concat, lit, floor, rand, explode, array

# 假设 key="hot_key" 数据量特别大
salt_buckets = 20  # 将热点 key 分散到 20 个分区

# Step 1: 给大表的 key 加随机盐
large_df_salted = large_df.withColumn(
    "salted_key",
    concat(col("join_key"), lit("_"), floor(rand() * salt_buckets).cast("string"))
)

# Step 2: 给小表扩展 salt_buckets 倍
salt_array = array([lit(str(i)) for i in range(salt_buckets)])

small_df_exploded = small_df.withColumn(
    "salt",
    explode(salt_array)
).withColumn(
    "salted_key",
    concat(col("join_key"), lit("_"), col("salt"))
).drop("salt")

# Step 3: 用 salted_key 进行 join
result = large_df_salted.join(small_df_exploded, "salted_key")

# Step 4: 清理（如果需要）
result = result.drop("salted_key")
```

**Salting 前后对比：**
```
Salting 前：
hot_key → 1个分区 → 1个Task处理10亿行 → 超级慢

Salting 后：
hot_key_0 → 分区0 → Task处理5000万行
hot_key_1 → 分区1 → Task处理5000万行
...
hot_key_19 → 分区19 → Task处理5000万行
→ 20个Task并行，快20倍
```

---



# Part 4: 实战必会高级特性

---



## AQE (Adaptive Query Execution) | 自适应查询执行

### 🎤 English Answer (30-40s)
"AQE is Spark 3.0's runtime optimization that adapts the query plan based on runtime statistics. Three key features: First, dynamic coalescing merges small partitions after shuffle to avoid overhead. Second, dynamic switching converts sort-merge join to broadcast join if one side turns out small. Third, skew join optimization detects and splits skewed partitions automatically. AQE makes Spark more self-tuning - you don't need to perfectly configure shuffle partitions or anticipate skew. Enable it with spark.sql.adaptive.enabled. It's a game-changer for production workloads."

### 📖 中文详解

**AQE 三大特性：**

### 1. 动态合并分区 (Coalesce Partitions)
```python
# 配置
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.minPartitionSize", "64MB")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128MB")

# 效果：shuffle 后自动合并小分区
# 原本：200 个分区，很多只有几 MB
# AQE 后：自动合并成 50 个分区，每个约 128MB
```

### 2. 动态切换 Join 策略
```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
# 运行时发现某张表 filter 后很小，自动切换到 broadcast join
```

### 3. 自动处理数据倾斜
```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", 5)
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256MB")

# 效果：检测到某分区是平均大小的 5 倍以上，自动拆分
```

**AQE 工作原理：**
```
传统执行：计划确定 → 执行所有 Stages
AQE 执行：执行 Stage 1 → 收集统计信息 → 重新优化后续计划 → 执行 Stage 2 → ...
```

---



## Dynamic Partition Pruning | 动态分区裁剪

### 🎤 English Answer (30-40s)
"Dynamic Partition Pruning or DPP optimizes star schema joins by pushing filter conditions from dimension tables to fact tables at runtime. In a typical query joining fact and dimension tables with a filter on dimension, DPP first executes the dimension query, collects the matching partition keys, then prunes fact table partitions before scanning. This can dramatically reduce fact table I/O. It works with partitioned tables and is enabled by default in Spark 3.0+. DPP is especially powerful for data warehouse queries where you filter by date or region."

### 📖 中文详解

**场景说明：**
```sql
-- 星型模型查询
SELECT f.amount, d.region
FROM fact_sales f
JOIN dim_store d ON f.store_id = d.store_id
WHERE d.region = 'West'

-- 没有 DPP：扫描整个 fact_sales 表
-- 有 DPP：只扫描 store_id 在 West 地区的分区
```

**工作原理：**
```
1. 执行维度表查询：SELECT store_id FROM dim_store WHERE region = 'West'
2. 收集结果：[101, 102, 105, ...]
3. 将这些值推送到事实表扫描：只读取 store_id IN (101, 102, 105, ...) 的分区
4. 执行 Join
```

**配置和使用：**
```python
# 启用 DPP（Spark 3.0+ 默认开启）
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true")
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.useStats", "true")
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.fallbackFilterRatio", "0.5")

# 确保事实表是分区表
# DPP 在分区列上效果最好
```

**查看是否使用了 DPP：**
```python
df.explain(True)
# 在物理计划中查找 "DynamicPruningExpression"
```

---



## Skew Join Optimization | 倾斜 Join 优化

### 🎤 English Answer (30-40s)
"Skew join optimization handles uneven data distribution during joins. In AQE, Spark detects skewed partitions during shuffle - partitions significantly larger than average. It then splits these large partitions and replicates corresponding data from the other side. Multiple tasks process the split data in parallel. You configure the detection threshold and factor. For manual optimization, techniques include salting keys, broadcasting small tables, or isolating skewed keys for separate processing. AQE's automatic handling works for many cases, but severe skew may still need manual salting."

### 📖 中文详解

**AQE 自动倾斜处理：**
```python
# 配置
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# 检测阈值：分区大小 > skewedPartitionFactor * 中位数大小
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", 5)

# 绝对阈值：分区大小 > 这个值才考虑
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256MB")
```

**AQE 处理流程：**
```
1. 执行 shuffle，收集分区大小统计
2. 识别倾斜分区（比如分区3特别大）
3. 将倾斜分区拆分成多个子分区
4. 将另一表的对应分区复制给每个子分区
5. 并行处理

示例：
原本：Partition 3 (10GB) join Partition 3' (100MB)
优化后：
  Split 3a (2GB) join Partition 3' (100MB)
  Split 3b (2GB) join Partition 3' (100MB)
  Split 3c (2GB) join Partition 3' (100MB)
  Split 3d (2GB) join Partition 3' (100MB)
  Split 3e (2GB) join Partition 3' (100MB)
→ 5个Task并行处理
```

**手动优化（当 AQE 不够时）：**
```python
# 方法1：SQL Hint
spark.sql("""
    SELECT /*+ SKEW('large_table', 'key', ('hot_value1', 'hot_value2')) */ *
    FROM large_table JOIN small_table ON large_table.key = small_table.key
""")

# 方法2：Salting（前面详细讲过）

# 方法3：隔离热点 Key
skewed_result = large_df.filter(col("key").isin(hot_keys)) \
                        .join(broadcast(small_df), "key")
normal_result = large_df.filter(~col("key").isin(hot_keys)) \
                        .join(small_df, "key")
final = skewed_result.union(normal_result)
```

---



# Part 5: 常用配置速查

---

## Key Spark Configurations | 关键配置速查

```python
# ========== 内存配置 ==========
spark.conf.set("spark.executor.memory", "8g")           # Executor 堆内存
spark.conf.set("spark.executor.memoryOverhead", "2g")   # Executor 堆外内存
spark.conf.set("spark.driver.memory", "4g")             # Driver 内存
spark.conf.set("spark.memory.fraction", "0.6")          # Spark 管理的内存比例
spark.conf.set("spark.memory.storageFraction", "0.5")   # Storage 初始比例

# ========== 并行度配置 ==========
spark.conf.set("spark.executor.cores", "4")             # 每个 Executor 的 cores
spark.conf.set("spark.default.parallelism", "200")      # RDD 默认并行度
spark.conf.set("spark.sql.shuffle.partitions", "200")   # Shuffle 后的分区数

# ========== Shuffle 配置 ==========
spark.conf.set("spark.shuffle.compress", "true")        # 压缩 shuffle 数据
spark.conf.set("spark.shuffle.spill.compress", "true")  # 压缩 spill 数据

# ========== AQE 配置 ==========
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128MB")

# ========== Join 配置 ==========
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")  # 10MB

# ========== 动态分区裁剪 ==========
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true")

# ========== 序列化 ==========
spark.conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")

# ========== 推测执行（处理慢 Task）==========
spark.conf.set("spark.speculation", "true")
spark.conf.set("spark.speculation.multiplier", "1.5")
```

---



# Quick Reference Card | 快速参考卡

```text
┌────────────────────────────────────────────────────────────────────┐
│                    SPARK INTERVIEW CHEAT SHEET                      │
├────────────────────────────────────────────────────────────────────┤
│ EXECUTION FLOW:                                                     │
│   Code → Driver → DAG → Stages → Tasks → Executors → Results       │
├────────────────────────────────────────────────────────────────────┤
│ DEPENDENCIES:                                                       │
│   Narrow: map, filter, select (no shuffle, same stage)             │
│   Wide: groupBy, join, repartition (shuffle, new stage)            │
├────────────────────────────────────────────────────────────────────┤
│ MEMORY MODEL:                                                       │
│   Executor Memory = Reserved + User + Spark (Execution + Storage)  │
├────────────────────────────────────────────────────────────────────┤
│ DATA SKEW SOLUTIONS:                                                │
│   1. Salting (add random prefix)                                   │
│   2. Broadcast join (small table)                                  │
│   3. AQE skew join (automatic)                                     │
│   4. Isolate hot keys (separate processing)                        │
├────────────────────────────────────────────────────────────────────┤
│ PARTITION SIZING:                                                   │
│   Target: 100-200MB per partition                                  │
│   Formula: Total Data Size / Target Size = Partition Count         │
├────────────────────────────────────────────────────────────────────┤
│ KEY OPTIMIZATIONS:                                                  │
│   1. Enable AQE                                                    │
│   2. Use broadcast joins                                           │
│   3. Right-size partitions                                         │
│   4. Cache wisely (unpersist when done)                            │
│   5. Prefer built-in functions over UDFs                           │
└────────────────────────────────────────────────────────────────────┘
```
